### Small example training BPE from start to finish

In [10]:
import regex as re
from collections import Counter, defaultdict
from typing import Tuple

In [11]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [12]:
re.findall(PAT, "some text that I'll pretokenize")

['some', ' text', ' that', ' I', "'ll", ' pretokenize']

In [13]:
for match in re.finditer(PAT, '“Rejoice with us,” said the air and the sunlight. “Enjoy thine own bright life in the fresh air.” \nBut the tree would not rejoice, though it grew taller every day; and, winter and summer, its dark-green foliage might be seen in the forest, while passers by would say, “What a beautiful tree!”'):
    print(match.group())

“
Rejoice
 with
 us
,”
 said
 the
 air
 and
 the
 sunlight
.
 “
Enjoy
 thine
 own
 bright
 life
 in
 the
 fresh
 air
.”
 


But
 the
 tree
 would
 not
 rejoice
,
 though
 it
 grew
 taller
 every
 day
;
 and
,
 winter
 and
 summer
,
 its
 dark
-
green
 foliage
 might
 be
 seen
 in
 the
 forest
,
 while
 passers
 by
 would
 say
,
 “
What
 a
 beautiful
 tree
!”


In [14]:
corpus = "low low low low low\nlower lower widest widest widest\nnewest newest newest newest newest newest <|endoftext|>"

In [17]:
pretokens = re.split(r'\s+', corpus)
c = Counter(pretokens)

# make sure to remove special tokens before beginning pretoken_freq table!!!
for special in special_tokens:
    del c[special]
print(c)

Counter({'newest': 6, 'low': 5, 'widest': 3, 'lower': 2})


In [18]:
l = ['ç'.encode('utf-8')]
print(tuple(l))

(b'\xc3\xa7',)


In [19]:
# from a word, creates a tuple of the literal bytes (discards char boundaries)
def word_to_tuple(word: str) -> Tuple[bytes]:
    encoded = word.encode("utf-8")
    t = [bytes([b]) for b in encoded]
    return tuple(t)

In [20]:
len(vocab)

257

In [21]:
# generate clean vocab to start
special_tokens = ['<|endoftext|>']
end_token = '<|endoftext|>'.encode('utf-8')

vocab = {}
for i in range(256):
    vocab[i] = bytes([i])
vocab[len(vocab)] = end_token
print(vocab)


# generate clean pretoken_freq table to start
pretoken_freq = {}
for pretoken in c:
    pretoken_freq[word_to_tuple(pretoken)] = c[pretoken]

print(f'Clean pretoken_freq:', pretoken_freq)

vocab_id = len(vocab)
num_merges = 6
for i in range(num_merges):

    # count freq of all pairs, keep track of max pair
    max_count = -1
    max_pair = None
    pair_freq = defaultdict(int)
    for pretoken in pretoken_freq:
        for pair_id in range(len(pretoken)-1):
            pair = (pretoken[pair_id], pretoken[pair_id+1])
            pair_freq[pair] += pretoken_freq[pretoken]
            if pair_freq[pair] >= max_count:
                if pair_freq[pair] == max_count and pair < max_pair:
                    continue
                else:
                    max_pair = pair
                    max_count = pair_freq[pair]
    
    # merge the max pair
    merge_pair = max_pair
    merge_pair_merged = bytes()
    for byte in merge_pair:
        merge_pair_merged += byte
    print(f'merged pair: {merge_pair_merged}')

    # add max pair to vocab
    vocab[vocab_id] = merge_pair_merged
    vocab_id += 1

    # keep track of changes, don't modify table directly
    changes = {}

    for pretoken in pretoken_freq:
        og_pretoken = pretoken
        curr_pretoken = pretoken
        pair_id = 0
        while pair_id < len(curr_pretoken)-1:
            pair = (curr_pretoken[pair_id], curr_pretoken[pair_id+1])
            if (pair == merge_pair):
                new_tuple = list(curr_pretoken[:pair_id])
                new_tuple.append(merge_pair_merged)
                rest_of_pretoken = list(curr_pretoken[pair_id + len(merge_pair):])
                new_tuple += rest_of_pretoken
                new_tuple = tuple(new_tuple)
                curr_pretoken = new_tuple
                changes[og_pretoken] = new_tuple
            pair_id += 1
    
    for old_key, new_key in changes.items():
        pretoken_freq[new_key] = pretoken_freq.pop(old_key)

print(pretoken_freq)

print(vocab)



{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

### Training BPE on TinyStories now

In [22]:
import regex as re
from collections import Counter, defaultdict
from typing import Tuple
# from pretokenization_example import find_chunk_boundaries

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [23]:
import os
from typing import BinaryIO


def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))


# ## Usage
# with open(..., "rb") as f:
#     num_processes = 4
#     boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

#     # The following is a serial implementation, but you can parallelize this
#     # by sending each start/end pair to a set of processes.
#     for start, end in zip(boundaries[:-1], boundaries[1:]):
#         f.seek(start)
#         chunk = f.read(end - start).decode("utf-8", errors="ignore")
#         # Run pre-tokenization on your chunk and store the counts for each pre-token


In [24]:
valid_path = '/juice5b/scr5b/kaitwang/cs336/data/TinyStoriesV2-GPT4-valid.txt'

In [25]:
with open(valid_path, "rb") as f:
    flag = 2
    num_desired_chunks = 10000
    boundaries = find_chunk_boundaries(f, num_desired_chunks, b"<|endoftext|>")

    # The following is a serial implementation, but you can parallelize this
    # by sending each start/end pair to a set of processes.
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        if flag > 0:
            if flag == 1:
                ex_chunk = chunk
            print(chunk)
            print('END OF CHUNK')
            flag -= 1
        # Run pre-tokenization on your chunk and store the counts for each pre-token

# print(boundaries)

u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.
Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."
Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."
They went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could hear t

#### pretokenize TinyStories, parallelized

In [26]:
from typing import Dict,Tuple

In [27]:
with open(valid_path, "rb") as f:
    num_desired_chunks = 32
    boundaries = find_chunk_boundaries(f, num_desired_chunks, b"<|endoftext|>")

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
     
        # Run pre-tokenization on your chunk and store the counts for each pre-token

# print(boundaries)

In [28]:
ex_chunk

'<|endoftext|>\n\nOnce upon a time there was a little girl named Lucy. She loved to go to the store to buy sweets with her mom and dad. On this special day, Lucy entered the store with her mom and dad, feeling so excited.\nAs they were looking around, Lucy noticed a little girl playing with a toy in the corner of the store. She gasped in excitement and ran towards her. Lucy asked if she could play too but the little girl said no. She was rather grumpy and was not in the mood to play.\nLucy\'s mom saw what was going on and told Lucy, "Let\'s try to be peaceful and kind to her. Have patience and understanding. Together, you can both be happy!"\nSo, Lucy smiled at the girl and said, "Can we play together?" The little girl softened and smiled back. She agreed to share the toy and even let Lucy have a turn first.\nLucy and the little girl played together happily. In the end, they both learnt an important lesson: be peaceful, kind, and understanding when faced with a conflict. And that is wh

### Tokenizing TinyStories

In [ ]:
import sys
import os
import torch
sys.path.append(os.path.dirname(os.path.abspath('.')))
from cs336_basics.tokenizer import Tokenizer
from cs336_basics.train_bpe import train_bpe
from cs336_basics.pretokenization_example import pretokenize_files_only
import time
import numpy as np
import pickle
from tqdm import tqdm
import multiprocessing

In [7]:
def generate_tinystories():
    device = torch.device('cpu')

    data_path = "/juice5b/scr5b/kaitwang/cs336/data"

    train_path = f"{data_path}/TinyStoriesV2-GPT4-train.txt"
    valid_path = f"{data_path}/TinyStoriesV2-GPT4-valid.txt"
    vocab_path = f"{data_path}/tinystories_vocab.pkl"
    merges_path = f"{data_path}/tinystories_merges.pkl"
    pretokens_train_path = f"{data_path}/tinystories_train_pretokens.npy"
    pretokens_valid_path = f"{data_path}/tinystories_valid_pretokens.npy"
    pretokens_internal_path = f"{data_path}/tinystories_internal.pkl"

    vocab_size = 10000
    special_tokens = ["<|endoftext|>"]

    # print("First pretokenize and save to file")
    pretokenize_file_only(
        input_path=train_path,
        pkl_output_path=pretokens_internal_path,
        max_workers = 64,
        special_tokens=special_tokens
    )

    print("Now BPE training...")
    vocab, merges = train_bpe(input_path = train_path, vocab_size = vocab_size, special_tokens = special_tokens, pretokens_internal_path=pretokens_internal_path)

    print("\nSaving vocabulary and merges...")
    tokenizer = Tokenizer(vocab=vocab, merges=merges, special_tokens=special_tokens)
    tokenizer.save(vocab_path, merges_path)
